# Question 1: [IPO] Withdrawn IPOs by Company Type

In [7]:
import pandas as pd
import numpy as np
import re
import requests
import io

**load data**

In [10]:
url = "https://www.iposcoop.com/ipos-recently-filed/"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

df = pd.read_html(io.StringIO(resp.text))[0]
df.head(3)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-24,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,26.25,$17.00,$17.00,$446.25,Priced,S/O
1,2026-09-24,Bluerock Acquisition Corp. II,BRRKU,BTIG,15.00,$10.00,$10.00,$150.00,Priced,S/O
2,2026-09-24,City Therapeutics,CTY,Goldman Sachs/Jefferies/Stifel/Oppenheimer & Co.,0.00,NaN,NaN,$100.00,TBA,S/O


In [11]:
df['Expected To Trade'].value_counts()

Expected To Trade
Priced       267
TBA          178
Withdrawn     34
Week of        7
Wednesday      5
Postponed      4
Thursday       2
Monday         1
Tuesday        1
Friday         1
Name: count, dtype: int64

In [14]:
# Фильтр: только Withdrawn, до 11.09.2026
df['File Date'] = pd.to_datetime(df['File Date'])
wd = df[(df['Expected To Trade'] == 'Withdrawn') &
       (df['File Date'] < '2026-09-11')].copy()

print(wd.shape)

(31, 10)


**company classification**

In [15]:
# --- Шаг 2: классификация (порядок правил важен — первое совпадение!) ---
def classify(name):
    clean = re.sub(r'\(.*?\)', '', str(name))
    if 'Technologies' in clean:
        return 'Technologies'
    if 'Acquisition Corp' in clean or 'Acquisition Corporation' in clean or 'Corp' in clean:
        return 'Acquisition Corp'
    if 'Inc' in clean or 'Incorporated' in clean:
        return 'Inc.'
    if 'Group' in clean:
        return 'Group'
    if 'Ltd' in clean or 'Limited' in clean:
        return 'Limited'
    if 'Holdings' in clean or 'Holding' in clean:
        return 'Holdings'
    return 'Other'

In [16]:
wd['Company Type'] = wd['Company'].apply(classify)
wd.sample(5)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type
334,2026-04-01,Going International Holding Company Limited,GIT,Prime Number Capital,0.0,NaN,NaN,$20.00,Withdrawn,S/O,Limited
491,2026-01-06,"Altech Digital Co., Ltd.",ALD,Pacific Century Securities/Revere Securities,1.5,$4.00,$4.00,$6.00,Withdrawn,S/O,Limited
52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.0,$4.00,$5.00,$22.50,Withdrawn,S/O,Technologies
462,2026-01-27,SFIDA X,SFDX,ThinkEquity,1.0,$5.00,$6.00,$5.50,Withdrawn,S/O,Other
196,2026-06-11,Club Versante Group Ltd.,CADV,Joseph Stone Capital,2.0,$4.00,$6.00,$10.00,Withdrawn,S/O,Group


**price parsing**

In [17]:
# --- Шаг 3: парсинг цен ---
def parse_price(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace('$', '').replace(',', '').strip()
    if s in ('-', ''):
        return np.nan
    m = re.search(r'\d+\.?\d*', s)
    return float(m.group()) if m else np.nan

In [21]:
wd['Avg_price'] = wd[['Price Low', 'Price High']].map(parse_price).mean(axis=1)
wd.sample(5)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Avg_price
244,2026-05-20,Vistek Limited (Withdrawn),VTEK,Bancroft Capital,6.25,$4.00,$6.00,$31.00,Withdrawn,S/O,Limited,5.0
185,2026-06-24,Kilcoy Global Foods Ltd. (Withdrawn),KGF,Citigroup/Morgan Stanley,0.00,NaN,NaN,$100.00,Withdrawn,S/O,Limited,NaN
260,2026-05-13,Ultratrex (Withdrawn),UTX,Craft Capital Management,1.25,$4.00,$6.00,$6.25,Withdrawn,S/O,Other,5.0
196,2026-06-11,Club Versante Group Ltd.,CADV,Joseph Stone Capital,2.00,$4.00,$6.00,$10.00,Withdrawn,S/O,Group,5.0
275,2026-05-05,EUPEC International Group Ltd.,EUPX,R.F. Lafferty & Co.,3.75,$4.00,$4.00,$15.00,Withdrawn,S/O,Group,4.0


**numeric cols**

In [22]:
# --- Шаг 4: числовые колонки ---
def parse_num(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace('$', '').replace(',', '').strip()
    return float(s) if s not in ('-', '') else np.nan

In [23]:
wd['Shares']  = wd['Shares (millions)'].apply(parse_num)
wd['Est_Vol'] = wd['Est $ Vol (millions)'].apply(parse_num)
wd.sample(5)

,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating,Company Type,Avg_price,Shares,Est_Vol
59,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,Roth Capital Partners/Stone X Financial Inc.,20.00,$10.00,$10.00,$200.00,Withdrawn,S/O,Acquisition Corp,10.0,20.00,200.00
186,2026-06-24,Safepoint Holdings (Withdrawn),SFPT,Deutsche Bank Securities/Morgan Stanley/Keefe ...,16.67,$15.00,$17.00,$266.72,Withdrawn,S/O,Holdings,16.0,16.67,266.72
52,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,Eddid Securities USA,5.00,$4.00,$5.00,$22.50,Withdrawn,S/O,Technologies,4.5,5.00,22.50
396,2026-02-19,Clear Street Group Inc. (Withdrawn),CLRS,Goldman Sachs/BofA Securities/Morgan Stanley/U...,13.00,$26.00,$28.00,$351.00,Withdrawn,S/O,Inc.,27.0,13.00,351.00
128,2026-07-31,Nuclea Energy (Withdrawn),NCLA,Joseph Gunnar & Co.,5.60,$8.00,$10.00,$50.40,Withdrawn,S/O,Other,9.0,5.60,50.40


**Value Calculation, aggregation**

In [24]:
wd['Shares_offered_value'] = wd['Shares'] * wd['Avg_price']
mask = wd['Shares_offered_value'].isna()
wd.loc[mask, 'Shares_offered_value'] = wd.loc[mask, 'Est_Vol']

result = (wd.groupby('Company Type')['Shares_offered_value']
          .sum().sort_values(ascending=False).round(2))
print(result)

Company Type
Acquisition Corp    499.98
Inc.                351.00
Holdings            311.66
Other               290.44
Limited             197.25
Technologies        184.90
Group                32.50
Name: Shares_offered_value, dtype: float64


# Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests, io

In [2]:
# --- Шаг 1: список IPO 2025 ---
url = "https://www.iposcoop.com/2025-pricings/"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
ipo = pd.read_html(io.StringIO(html))[0]          # 231 IPO

In [3]:
# --- Шаг 2: фильтрация ---
ipo['Offer Date'] = pd.to_datetime(ipo['Offer Date'])
ipo['Return_num'] = ipo['Return'].str.replace('%', '').astype(float)
filtered = ipo[(ipo['Offer Date'] < '2025-09-01') & (ipo['Return_num'] != 0)]
tickers = filtered['Symbol'].tolist()             # ~148 штук

In [4]:
# --- Шаг 3: выгрузка OHLCV ---
frames = []
for t in tickers:
    d = yf.download(t, start='2025-01-01', end='2026-09-12',  # end — НЕ включительно!
                    progress=False, auto_adjust=False)
    if d is None or d.empty:
        continue                                   # делистинг / нет в Yahoo
    if isinstance(d.columns, pd.MultiIndex):
        d.columns = d.columns.get_level_values(0)
    d['Ticker'] = t
    frames.append(d.reset_index())
stocks_df = pd.concat(frames, ignore_index=True) \
              .sort_values(['Ticker', 'Date']).reset_index(drop=True)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: possibly delisted; no timezone found

1 Failed download:
['MJID']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
$EMPG: possibly delisted; no timezone found

1 Failed download:
['EMPG']: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found

1 Failed download:
['CAEP']: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found

1 Failed download:
['PTNM']: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found

1 Failed download:
['AHL']: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found

1 Failed download:
['CEPT']: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found

1 Failed download:
['SDM']: possibly d

In [5]:
# --- Шаг 4: фичи (строго по формулам задания) ---
g = stocks_df.groupby('Ticker')['Close']
stocks_df['growth_252d'] = stocks_df['Close'] / g.shift(252)
stocks_df['volatility']  = g.transform(lambda s: s.rolling(30).std()) * np.sqrt(252)

In [6]:
# --- Шаг 5: Sharpe, risk-free = 5% ---
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

# --- Шаг 6: срез на 2026-09-11 ---
day = stocks_df[stocks_df['Date'] == '2026-09-11'].copy()
day['Sharpe'] = day['Sharpe'].replace([np.inf, -np.inf], np.nan)  # volatility=0
print(day[['growth_252d', 'volatility', 'Sharpe']].describe())
print("Медианный Sharpe:", day['Sharpe'].median())

Price  growth_252d  volatility        Sharpe
count   130.000000  131.000000  1.280000e+02
mean      1.052117   27.571999  3.277917e+04
std       3.049217   66.868036  3.708519e+05
min       0.001005    0.000000 -4.014736e-02
25%       0.141035    2.301309  1.130346e-02
50%       0.594534    7.885748  4.722044e-02
75%       1.038821   23.968537  1.237578e-01
max      33.638270  627.698106  4.195710e+06
Медианный Sharpe: 0.047220436195126426


# Question 3: [IPO] 'Fixed Months Holding Strategy'

In [7]:
# --- Шаг 1: future growth на 1..12 месяцев (1 мес = 21 торговый день) ---
g = stocks_df.groupby('Ticker')['Close']
growth_cols = [f'future_growth_{m}_m' for m in range(1, 13)]
for m, col in zip(range(1, 13), growth_cols):
    stocks_df[col] = g.shift(-21 * m) / stocks_df['Close']

In [8]:
# --- Шаг 2: первый торговый день каждого тикера ---
min_dates = (stocks_df.groupby('Ticker', as_index=False)['Date'].min()
             .rename(columns={'Date': 'min_date'}))

# --- Шаг 3: inner join -> одна строка на тикер (день IPO) ---
entry = stocks_df.merge(min_dates,
                        left_on=['Ticker', 'Date'],
                        right_on=['Ticker', 'min_date'],
                        how='inner')
print(entry.shape)

(132, 24)


In [9]:
# --- Шаг 4: описательная статистика и оптимальный месяц ---
stats = entry[growth_cols].describe()
print(stats.T[['count', 'mean', '50%']])

medians = stats.loc['50%']
best_col = medians.idxmax()
print(f"Оптимально: {growth_cols.index(best_col) + 1} мес., медиана = {medians.max():.4f}")

                    count       mean       50%
future_growth_1_m   132.0  95.751507  0.935351
future_growth_2_m   131.0  67.978625  0.892958
future_growth_3_m   131.0  51.247805  0.827160
future_growth_4_m   131.0  22.697288  0.730400
future_growth_5_m   131.0  54.242665  0.690909
future_growth_6_m   131.0  64.010844  0.700916
future_growth_7_m   131.0  59.265533  0.659973
future_growth_8_m   131.0  21.457209  0.603508
future_growth_9_m   131.0  20.006901  0.580294
future_growth_10_m  131.0  11.256621  0.533333
future_growth_11_m  131.0   8.899609  0.481811
future_growth_12_m  130.0   5.818939  0.491838
Оптимально: 1 мес., медиана = 0.9354


# Question 4: [Strategy] Simple RSI-Based Trading Strategy

In [1]:
# --- Шаг 1: загрузка parquet с индикаторами ---
import gdown
import pandas as pd
from pathlib import Path

file_id = "1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-"
if not Path("data.parquet").exists():
    gdown.download(f"https://drive.google.com/uc?id={file_id}", "data.parquet", quiet=False)
df = pd.read_parquet("data.parquet", engine="pyarrow")
print(df.shape)

(229932, 203)


In [ ]:
# --- Шаг 2-3: сигнал RSI < 30 в окне 2000-01-01 .. 2025-06-01 ---
RSI_THRESHOLD = 30
selected_df = df[(df['rsi'] < RSI_THRESHOLD) &
                 (df['Date'] >= '2000-01-01') &
                 (df['Date'] <= '2025-06-01')]

# --- Шаг 4: $1000 на каждый сигнал, выход через 30 дней ---
ret = selected_df['growth_future_30d'] - 1
net_income = 1000 * ret.sum()   

print(f"Сделок: {len(selected_df)}")
print(f"Средняя доходность 30d: {ret.mean():.2%}, win rate: {(ret > 0).mean():.2%}")
print(f"Net income: ${net_income:,.0f} (~{net_income / 1000:.1f}K)")

Сделок: 5206
Средняя доходность 30d: 1.26%, win rate: 55.13%
Net income: $65,806 (~65.8K)


# Q5. [Exploratory, Optional] Predicting a Positive-Return IPO